# Modified GASky + EfficientNetB0 + SVM — validation-tuned, image-level split

Eksperimen klasifikasi citra sitologi serviks (NILM, LSIL, HSIL, SCC). Notebook membandingkan EfficientNetB0→GAP→SVM dengan adaptasi Modified GASky dan beberapa ablation. Quantile proximity, cap region, serta C/gamma SVM dipilih pada validation; test digunakan untuk evaluasi akhir dan perbandingan komponen yang telah ditetapkan.

Dataset Mendeley LBC dilaporkan berasal dari 460 pasien, tetapi distribusi file yang digunakan tidak memuat patient/slide ID untuk group split. Protokol karena itu stratified image-level split 70:15:15 dan tidak menjamin independensi antar pasien.

GASky asli memilih grid geografis berdasarkan interval jarak ke fasilitas. Notebook mengadaptasi idenya ke region feature map dan anchor kelas; ini metode usulan yang perlu dievaluasi, bukan implementasi GASky identik. Jalankan seluruh sel dari awal di Colab untuk menghasilkan tabel sweep, ablation, bootstrap CI, metrik, dan latensi.


## 0. Persiapan environment dan import

In [ ]:
# Google Colab: mount Google Drive
from pathlib import Path
import os, random, hashlib, json, time, math, warnings
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             precision_recall_fscore_support)
import joblib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Google Colab tidak terdeteksi; pastikan path dataset tersedia.')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 1. Konfigurasi dataset

In [ ]:
# Path utama mengikuti konfigurasi notebook sebelumnya
DATA_DIR = Path('/content/drive/MyDrive/dataset_cervical')

# Mendukung dua struktur folder yang pernah digunakan
CLASS_NAMES = ['NILM', 'LSIL', 'HSIL', 'SCC']
CANDIDATE_ROOTS = [DATA_DIR / 'dataset_lbc', DATA_DIR]
IMAGE_ROOT = next((p for p in CANDIDATE_ROOTS
                   if all((p / c).is_dir() for c in CLASS_NAMES)), None)
if IMAGE_ROOT is None:
    raise FileNotFoundError(
        f'Folder kelas tidak ditemukan. Periksa {DATA_DIR}.\n'
        'Struktur yang didukung: dataset_cervical/dataset_lbc/{NILM,LSIL,HSIL,SCC} '
        'atau dataset_cervical/{NILM,LSIL,HSIL,SCC}'
    )

OUTPUT_DIR = DATA_DIR / 'outputs_modified_gasky_image_split'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 224
BATCH_SIZE = 16
REGION_H, REGION_W = 2, 2
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15  # fraksi dari seluruh dataset; dibentuk setelah test dipisah
AUGMENT_MAX_MULTIPLIER = 3  # batasi jumlah hasil augmentasi per kelas

# Modified GASky: batas kedekatan DESIRED dikalibrasi hanya dari TRAIN.
# Quantile lebih kecil = seleksi lebih ketat.
DESIRED_PROXIMITY_QUANTILE = 0.50
# Batas pengaman agar skyline tidak mengambil hampir semua region.
# Jika skyline lebih besar dari batas ini, dipilih kandidat terbaik berdasarkan
# jarak desired (utama), lalu jarak undesired (pemecah seri).
MAX_SELECTED_REGION_FRACTION = 0.50

print('DATA_DIR   :', DATA_DIR)
print('IMAGE_ROOT :', IMAGE_ROOT)
print('OUTPUT_DIR :', OUTPUT_DIR)


## 2. Audit dataset asli dan pemeriksaan duplikasi exact

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
rows = []
for label in CLASS_NAMES:
    folder = IMAGE_ROOT / label
    for p in sorted(folder.rglob('*')):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            rows.append({'filepath': str(p.resolve()), 'label': label})
data = pd.DataFrame(rows)
if data.empty:
    raise ValueError(f'Tidak ada file citra ditemukan di {IMAGE_ROOT}')
data['y'] = data['label'].map({c:i for i,c in enumerate(CLASS_NAMES)}).astype(int)

print('Jumlah citra ditemukan:', len(data))
counts = data['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
display(counts.rename('jumlah_citra').to_frame())

plt.figure(figsize=(8,4))
plt.bar(counts.index, counts.values)
plt.title('Distribusi kelas sebelum splitting')
plt.xlabel('Kelas'); plt.ylabel('Jumlah citra'); plt.grid(axis='y', alpha=.25)
plt.show()

# Hash file identik: duplikat dengan label berbeda perlu diperiksa manual.
def sha256_file(path, chunk_size=1<<20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

t0 = time.time()
data['sha256'] = data['filepath'].map(sha256_file)
conflicts = data.groupby('sha256')['label'].nunique()
conflict_hashes = conflicts[conflicts > 1].index
if len(conflict_hashes):
    display(data[data.sha256.isin(conflict_hashes)].sort_values('sha256'))
    raise ValueError('Ada file identik dengan label berbeda. Periksa label sebelum melanjutkan.')

n_before = len(data)
data = data.drop_duplicates('sha256', keep='first').reset_index(drop=True)
print(f'Duplikasi exact yang dibuang: {n_before-len(data)}')
print(f'Waktu audit hash: {time.time()-t0:.1f} detik')
print('Jumlah citra setelah deduplikasi:', len(data))


## 3. Image-level stratified split 70:15:15

Split dilakukan pada **citra asli sebelum preprocessing dan augmentasi**. Stratifikasi menjaga proporsi kelas sedekat mungkin. Deduplikasi exact dilakukan sebelum split agar file identik tidak tersebar antar-subset. Pemeriksaan ini tidak menjamin bebas *near-duplicate* atau bebas kebocoran antar-citra dari pasien yang sama.

In [ ]:
# Sisihkan 15% test, lalu sisihkan 15% dari keseluruhan sebagai validation.
trainval_df, test_df = train_test_split(
    data, test_size=TEST_FRACTION, random_state=SEED,
    stratify=data['y']
)
val_fraction_of_trainval = VAL_FRACTION / (1.0 - TEST_FRACTION)
train_df, val_df = train_test_split(
    trainval_df, test_size=val_fraction_of_trainval,
    random_state=SEED, stratify=trainval_df['y']
)

for name, part in [('TRAIN', train_df), ('VALIDATION', val_df), ('TEST', test_df)]:
    print(f'{name}: {len(part)} citra ({len(part)/len(data):.1%})')
    display(part['label'].value_counts().reindex(CLASS_NAMES, fill_value=0).rename(name).to_frame())

# Visualisasi distribusi kelas pada setiap subset
split_counts = pd.DataFrame({
    'Train': train_df.label.value_counts().reindex(CLASS_NAMES, fill_value=0),
    'Validation': val_df.label.value_counts().reindex(CLASS_NAMES, fill_value=0),
    'Test': test_df.label.value_counts().reindex(CLASS_NAMES, fill_value=0)
})
ax = split_counts.plot(kind='bar', figsize=(9,4))
ax.set_title('Distribusi kelas setelah image-level split')
ax.set_xlabel('Kelas'); ax.set_ylabel('Jumlah citra'); ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# Pastikan tidak ada filepath yang berulang antar subset
paths = [set(x.filepath) for x in (train_df, val_df, test_df)]
assert not (paths[0]&paths[1] or paths[0]&paths[2] or paths[1]&paths[2])
print('Pemeriksaan filepath overlap: LULUS')


## 4. Load dan preprocessing citra

In [ ]:
# Load citra RGB dan resize ke 224x224. Nilai piksel tetap 0..255.
# EfficientNetB0 Keras memiliki preprocessing/rescaling internal.
def read_image(path, image_size=IMAGE_SIZE):
    img = Image.open(path).convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def load_images(frame):
    images = np.stack([read_image(p) for p in frame.filepath])
    labels = frame.y.to_numpy(dtype=np.int64)
    return images, labels

X_train_img, y_train = load_images(train_df)
X_val_img, y_val = load_images(val_df)
X_test_img, y_test = load_images(test_df)
print('Shape train/val/test:', X_train_img.shape, X_val_img.shape, X_test_img.shape)

# Tampilkan beberapa contoh citra asli
fig, axes = plt.subplots(2, 4, figsize=(12,6))
for ax, idx in zip(axes.flat, np.linspace(0, len(X_train_img)-1, 8, dtype=int)):
    ax.imshow(X_train_img[idx]); ax.set_title(CLASS_NAMES[y_train[idx]]); ax.axis('off')
plt.suptitle('Contoh citra TRAIN setelah resize'); plt.tight_layout(); plt.show()


## 5. Penanganan class imbalance — augmentasi TRAIN saja

Augmentasi dilakukan **setelah split** dan hanya pada TRAIN. Validation dan test tetap citra asli. Untuk menghindari augmentasi berlebihan, jumlah citra per kelas ditingkatkan paling banyak sampai `min(jumlah kelas terbesar, 3 × jumlah kelas tersebut)`. `class_weight='balanced'` pada SVM tetap digunakan sebagai penanganan tambahan.

Transformasi yang digunakan bersifat ringan: flip horizontal/vertikal, rotasi kecil, zoom kecil, dan contrast.

In [ ]:
augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    tf.keras.layers.RandomRotation(0.08, fill_mode='reflect', seed=SEED+1),
    tf.keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED+2),
    tf.keras.layers.RandomContrast(0.10, seed=SEED+3),
], name='train_only_augmenter')

train_counts = pd.Series(y_train).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
max_count = int(train_counts.max())
aug_target = {k: min(max_count, int(train_counts[k] * AUGMENT_MAX_MULTIPLIER)) for k in range(len(CLASS_NAMES))}

aug_images = [img for img in X_train_img]
aug_labels = list(y_train)
aug_origin = ['original'] * len(X_train_img)

for k in range(len(CLASS_NAMES)):
    idxs = np.where(y_train == k)[0]
    n_add = max(0, aug_target[k] - len(idxs))
    for j in range(n_add):
        source = X_train_img[idxs[j % len(idxs)]]
        x = tf.convert_to_tensor(source[None, ...], dtype=tf.float32)
        x_aug = augmenter(x, training=True)[0]
        x_aug = tf.clip_by_value(x_aug, 0, 255)
        aug_images.append(tf.cast(x_aug, tf.uint8).numpy())
        aug_labels.append(k)
        aug_origin.append('augmented')

X_train_aug = np.stack(aug_images)
y_train_aug = np.asarray(aug_labels, dtype=np.int64)
aug_origin = np.asarray(aug_origin)

before = pd.Series(y_train).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
after = pd.Series(y_train_aug).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
imbalance_table = pd.DataFrame({'train_asli': before.values, 'train_setelah_aug': after.values}, index=CLASS_NAMES)
display(imbalance_table)

ax = imbalance_table.plot(kind='bar', figsize=(9,4))
ax.set_title('Jumlah citra TRAIN sebelum dan sesudah augmentasi')
ax.set_xlabel('Kelas'); ax.set_ylabel('Jumlah citra'); ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# Tampilkan pasangan citra asli dan hasil augmentasi untuk setiap kelas
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flat

for k in range(len(CLASS_NAMES)):
    # Ambil indeks citra TRAIN asli dari kelas k
    original_idx = np.where(y_train == k)[0][0]

    # Cari citra hasil augmentasi pertama dari kelas k
    class_indices = np.where(y_train_aug == k)[0]
    aug_idx = next(
        (i for i in class_indices if aug_origin[i] == 'augmented'),
        class_indices[0]
    )

    # Posisi subplot: dua kolom untuk setiap kelas
    ax_original = axes[2 * k]
    ax_augmented = axes[2 * k + 1]

    # Citra asli
    ax_original.imshow(X_train_img[original_idx])
    ax_original.set_title(f'{CLASS_NAMES[k]} asli')
    ax_original.axis('off')

    # Citra augmentasi
    ax_augmented.imshow(X_train_aug[aug_idx])
    title = (
        f'{CLASS_NAMES[k]} augmentasi'
        if aug_origin[aug_idx] == 'augmented'
        else f'{CLASS_NAMES[k]} (tanpa augmentasi)'
    )
    ax_augmented.set_title(title)
    ax_augmented.axis('off')

plt.suptitle('Contoh augmentasi TRAIN')
plt.tight_layout()
plt.show()


## 6. Ekstraksi feature map EfficientNetB0

In [ ]:
# Backbone pretrained ImageNet, dibekukan sebagai feature extractor.
backbone_base = EfficientNetB0(include_top=False, weights='imagenet',
                               input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
backbone_base.trainable = False
feature_extractor = tf.keras.Model(backbone_base.input, backbone_base.output,
                                   name='efficientnetb0_feature_extractor')
print('Feature map output shape:', feature_extractor.output_shape)

# Pastikan input dalam rentang 0..255; preprocessing internal EfficientNet menangani rescaling.
def extract_maps(images, batch_size=BATCH_SIZE):
    outputs = []
    for start in range(0, len(images), batch_size):
        batch = images[start:start+batch_size].astype(np.float32)
        fmap = feature_extractor(batch, training=False).numpy().astype(np.float32)
        outputs.append(fmap)
    return np.concatenate(outputs, axis=0)

t0 = time.time()
train_maps = extract_maps(X_train_aug)
val_maps = extract_maps(X_val_img)
test_maps = extract_maps(X_test_img)
print('Train feature maps:', train_maps.shape)
print('Validation feature maps:', val_maps.shape)
print('Test feature maps:', test_maps.shape)
print(f'Waktu ekstraksi: {time.time()-t0:.1f} detik')

# Visualisasi feature map: rata-rata aktivasi lintas channel untuk satu contoh
sample_map = train_maps[0]
plt.figure(figsize=(5,4))
plt.imshow(sample_map.mean(axis=-1), cmap='viridis')
plt.colorbar(label='Mean activation')
plt.title(f'Rata-rata aktivasi feature map — {sample_map.shape}')
plt.axis('off'); plt.show()


## 7. Standardisasi channel dan pembentukan anchor kelas

Setiap posisi spasial pada feature map merupakan descriptor berdimensi `C`. StandardScaler channel di-fit hanya pada descriptor dari TRAIN. Untuk setiap citra TRAIN, descriptor spasial dirata-ratakan (GAP) menjadi vektor `g_i`. Anchor kelas `a_k` adalah rata-rata `g_i` dari citra TRAIN kelas `k`.

In [ ]:
def fit_channel_scaler(train_feature_maps):
    c = train_feature_maps.shape[-1]
    scaler = StandardScaler()
    for fmap in train_feature_maps:
        scaler.partial_fit(fmap.reshape(-1, c))
    return scaler

def apply_channel_scaler(feature_maps, scaler):
    n,h,w,c = feature_maps.shape
    flat = feature_maps.reshape(-1,c)
    return scaler.transform(flat).reshape(n,h,w,c).astype(np.float32)

channel_scaler = fit_channel_scaler(train_maps)
train_maps_s = apply_channel_scaler(train_maps, channel_scaler)
val_maps_s = apply_channel_scaler(val_maps, channel_scaler)
test_maps_s = apply_channel_scaler(test_maps, channel_scaler)

# Anchor dihitung dari data TRAIN (termasuk augmentasi TRAIN saja).
def build_class_anchors(feature_maps, labels):
    image_vectors = feature_maps.mean(axis=(1,2))
    anchors = {}
    for k in range(len(CLASS_NAMES)):
        if not np.any(labels == k):
            raise ValueError(f'Tidak ada sampel TRAIN untuk kelas {CLASS_NAMES[k]}')
        anchors[k] = image_vectors[labels == k].mean(axis=0)
    return anchors

anchors = build_class_anchors(train_maps_s, y_train_aug)
print('Jumlah anchor:', len(anchors))
print('Dimensi setiap anchor:', next(iter(anchors.values())).shape)

## 8. Modified GASky — region, interval, proximity gate, dan skyline

Setiap region diringkas sebagai interval jarak `[d_min, d_max]` terhadap empat anchor kelas. Untuk query `q`, anchor kelas `q` adalah **desired** (lebih dekat lebih baik), sedangkan tiga anchor lain adalah **undesired** (lebih jauh lebih baik).

### Mengapa ditambahkan proximity gate?

Pada Area Skyline asli, semua area yang tidak didominasi dapat masuk skyline. Artinya, sebuah area yang sangat jauh dari fasilitas desirable masih mungkin bertahan hanya karena tidak ada area lain yang mendominasinya. Paper Annisa et al. (2016), bagian 3.1.4 dan 3.2.3 (hlm. 949–950), mendefinisikan skyline melalui dominansi min–max; paper juga mencatat bahwa area besar dapat sulit didominasi dan meningkatkan rasio skyline (bagian 3.3–4.2, hlm. 951–953).

Untuk kebutuhan penelitian ini, **non-dominated tidak otomatis berarti relevan**. Karena itu, Modified GASky menambahkan dua tahap:

1. **Desired proximity gate:** region hanya menjadi kandidat jika `d_max(region, anchor_desired) ≤ τ_q`. Memakai `d_max` berarti seluruh descriptor di dalam region harus berada dalam batas kedekatan desired, bukan hanya satu titik yang kebetulan dekat.
2. **Skyline pada kandidat yang lolos:** dominansi interval desired/undesired diterapkan hanya pada region yang lolos gate.
3. **Batas jumlah region:** jika skyline masih melebihi fraksi maksimum, dipangkas secara deterministik: prioritas pertama `d_max` desired terkecil; jika seri, `d_min` undesired rata-rata terbesar.

Ambang `τ_q` dihitung dari kuantil distribusi `d_max` region TRAIN untuk query `q` (default median/50%). Ambang tidak dihitung dari validation/test. Kuantil dan fraksi maksimum adalah hyperparameter eksperimen yang dapat dibandingkan melalui validation; jangan memilihnya berdasarkan test.

Jika tidak ada region yang lolos gate pada suatu citra, satu region dengan `d_max` desired terkecil dipakai sebagai fallback agar vektor fitur tetap terdefinisi. Kondisi fallback dicatat sebagai diagnostik.

**Ini adalah modifikasi yang diusulkan, bukan langkah yang diklaim terdapat pada GASky asli.**

**Mengapa memakai `d_max` pada gate, bukan `d_min`?** `d_min` hanya membuktikan ada setidaknya satu descriptor yang dekat dengan desired. Sebuah region yang mayoritas isinya jauh masih dapat lolos. `d_max ≤ τ_q` lebih konservatif: seluruh descriptor dalam region harus berada di dalam batas kedekatan yang ditetapkan. Ini sejalan dengan maksud area/region sebagai satu unit seleksi.


In [ ]:
def make_regions(fmap, region_h=REGION_H, region_w=REGION_W):
    """Membagi feature map menjadi blok non-overlapping; mengembalikan descriptor dan koordinat."""
    h, w, c = fmap.shape
    regions, coords = [], []
    for y0 in range(0, h, region_h):
        for x0 in range(0, w, region_w):
            y1, x1 = min(y0 + region_h, h), min(x0 + region_w, w)
            block = fmap[y0:y1, x0:x1, :]
            regions.append(block.reshape(-1, c))
            coords.append((y0, y1, x0, x1))
    return regions, coords


def interval_to_anchor(descriptors, anchor):
    """Interval [minimum, maximum] jarak descriptor region terhadap satu anchor."""
    distances = np.linalg.norm(descriptors - anchor[None, :], axis=1)
    return float(distances.min()), float(distances.max())


def dominates(interval_a, interval_b, desired_idx):
    """
    Dominansi interval untuk query kelas desired_idx.
    Desired: A lebih dekat jika max(A) <= min(B).
    Undesired: A lebih jauh jika min(A) >= max(B).
    """
    strict = False
    for k in range(len(CLASS_NAMES)):
        a_min, a_max = interval_a[k]
        b_min, b_max = interval_b[k]
        if k == desired_idx:
            if not (a_max <= b_min):
                return False
            strict |= (a_max < b_min)
        else:
            if not (a_min >= b_max):
                return False
            strict |= (a_min > b_max)
    return bool(strict)


def skyline_indices(intervals, desired_idx):
    """Skyline non-dominated dari kandidat yang sudah lolos desired gate."""
    n = len(intervals)
    keep = []
    for j in range(n):
        is_dominated = any(
            i != j and dominates(intervals[i], intervals[j], desired_idx)
            for i in range(n)
        )
        if not is_dominated:
            keep.append(j)
    return keep


def fit_desired_proximity_thresholds(train_feature_maps, anchors,
                                     quantile=DESIRED_PROXIMITY_QUANTILE):
    """
    Kalibrasi tau_q dari TRAIN saja.
    Untuk setiap query q, kumpulkan d_max semua region TRAIN ke anchor desired q.
    """
    distance_samples = {q: [] for q in range(len(CLASS_NAMES))}
    for fmap in train_feature_maps:
        regions, _ = make_regions(fmap)
        for q in range(len(CLASS_NAMES)):
            for descriptors in regions:
                _, d_max = interval_to_anchor(descriptors, anchors[q])
                distance_samples[q].append(d_max)

    thresholds = {}
    for q, values in distance_samples.items():
        if not values:
            raise ValueError(f'Tidak ada region TRAIN untuk query {CLASS_NAMES[q]}')
        thresholds[q] = float(np.quantile(values, quantile))
    return thresholds, distance_samples


def cap_skyline_candidates(candidate_indices, intervals, desired_idx,
                           max_fraction=MAX_SELECTED_REGION_FRACTION,
                           total_regions=None):
    """
    Batas pengaman jumlah region. Jika skyline melampaui batas,
    urutkan berdasarkan d_max desired terkecil, lalu rata-rata d_min undesired terbesar.
    """
    if total_regions is None:
        total_regions = len(intervals)
    max_keep = max(1, int(math.ceil(total_regions * max_fraction)))
    if len(candidate_indices) <= max_keep:
        return list(candidate_indices), False

    undesired_idx = [k for k in range(len(CLASS_NAMES)) if k != desired_idx]
    ranked = sorted(
        candidate_indices,
        key=lambda ridx: (
            float(intervals[ridx, desired_idx, 1]),
            -float(intervals[ridx, undesired_idx, 0].mean())
        )
    )
    return ranked[:max_keep], True


def transform_one_image(fmap, anchors, desired_thresholds,
                        return_details=False, use_gate=True,
                        use_skyline=True, use_cap=True,
                        max_selected_fraction=None):
    """
    Per query:
      A. desired proximity gate,
      B. skyline hanya pada kandidat yang lolos,
      C. cap jumlah region bila skyline terlalu besar,
      D. agregasi Mean + Max.
    """
    regions, coords = make_regions(fmap)
    n_regions = len(regions)
    intervals = np.zeros((n_regions, len(CLASS_NAMES), 2), dtype=np.float32)

    for ridx, descriptors in enumerate(regions):
        for k in range(len(CLASS_NAMES)):
            intervals[ridx, k] = interval_to_anchor(descriptors, anchors[k])

    q_features, q_selected = [], []
    diagnostics = []

    for q in range(len(CLASS_NAMES)):
        # Gate berdasarkan d_max desired: seluruh descriptor region harus cukup dekat.
        if use_gate:
            eligible = np.where(intervals[:, q, 1] <= desired_thresholds[q])[0].tolist()
        else:
            eligible = list(range(n_regions))

        fallback_used = False
        if not eligible:
            eligible = [int(np.argmin(intervals[:, q, 1]))]
            fallback_used = True

        if use_skyline:
            local_skyline = skyline_indices(intervals[eligible], q)
            skyline_global = [eligible[i] for i in local_skyline]
        else:
            skyline_global = list(eligible)

        if use_cap:
            cap_fraction = (MAX_SELECTED_REGION_FRACTION if max_selected_fraction is None
                            else max_selected_fraction)
            selected, cap_applied = cap_skyline_candidates(
                skyline_global, intervals, q,
                max_fraction=cap_fraction,
                total_regions=n_regions
            )
        else:
            selected, cap_applied = list(skyline_global), False

        selected_desc = np.concatenate([regions[i] for i in selected], axis=0)
        feat_mean = selected_desc.mean(axis=0)
        feat_max = selected_desc.max(axis=0)
        q_features.append(np.concatenate([feat_mean, feat_max]))
        q_selected.append(selected)

        diagnostics.append({
            'query': CLASS_NAMES[q],
            'n_total': n_regions,
            'n_eligible': len(eligible),
            'n_skyline_before_cap': len(skyline_global),
            'n_selected': len(selected),
            'fallback_used': fallback_used,
            'cap_applied': cap_applied,
            'desired_tau': float(desired_thresholds[q])
        })

    feature = np.concatenate(q_features)
    if return_details:
        return feature, q_selected, coords, intervals, diagnostics
    return feature, [len(x) for x in q_selected]

### Kalibrasi ambang desired dari TRAIN
Fungsi pada bagian sebelumnya kini digunakan untuk menghitung ambang `τ_q` dari distribusi `d_max` region TRAIN. Ambang yang sama dipakai untuk validation dan test.

In [ ]:
# Kalibrasi proximity threshold hanya dari TRAIN.
desired_thresholds, train_desired_distance_samples = fit_desired_proximity_thresholds(
    train_maps_s, anchors, quantile=DESIRED_PROXIMITY_QUANTILE
)
print('Desired proximity thresholds (TRAIN only):')
display(pd.DataFrame({
    'query': CLASS_NAMES,
    'tau_q': [desired_thresholds[q] for q in range(len(CLASS_NAMES))],
    'quantile': [DESIRED_PROXIMITY_QUANTILE] * len(CLASS_NAMES)
}))

# Visualisasi distribusi d_max desired TRAIN dan ambang tiap query.
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(16, 3.5))
for q, ax in enumerate(axes):
    vals = train_desired_distance_samples[q]
    ax.hist(vals, bins=35, alpha=0.8)
    ax.axvline(desired_thresholds[q], color='red', linestyle='--',
               label=f'tau={desired_thresholds[q]:.2f}')
    ax.set_title(f'{CLASS_NAMES[q]} desired')
    ax.set_xlabel('d_max region → anchor')
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8)
plt.suptitle('Kalibrasi proximity gate dari TRAIN saja')
plt.tight_layout()
plt.show()


# Uji dimensi fitur dan konsistensi seleksi.
example_feature, example_selected, example_coords, example_intervals, example_diag = transform_one_image(
    train_maps_s[0], anchors, desired_thresholds, return_details=True
)
C = train_maps_s.shape[-1]
print('Feature map H×W×C:', train_maps_s.shape[1:])
print('Jumlah region:', len(example_coords))
print('Dimensi fitur Modified GASky:', len(example_feature), '| target 8×C =', 8*C)
print('Jumlah region terpilih per query:', [len(x) for x in example_selected])
assert len(example_feature) == 8*C
assert all(1 <= len(x) <= max(1, math.ceil(len(example_coords)*MAX_SELECTED_REGION_FRACTION))
           for x in example_selected)


## 9. Visualisasi region terpilih dan region yang dieliminasi

Visualisasi membedakan tiga status: hijau = region final terpilih; kuning = lolos desired proximity gate tetapi tidak menjadi region final; merah = gagal gate karena terlalu jauh dari anchor desired. Ini membantu memeriksa apakah filter bekerja sesuai maksud, bukan sekadar menampilkan skyline akhir.

In [ ]:
# Visualisasi status region untuk setiap query pada satu contoh TRAIN.
# Hijau = terpilih; kuning = lolos proximity gate tetapi tidak masuk skyline/final;
# merah = gagal desired proximity gate.
example_feature, example_selected, example_coords, example_intervals, example_diag = transform_one_image(
    train_maps_s[0], anchors, desired_thresholds, return_details=True
)

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(17, 4))
for q, ax in enumerate(axes):
    ax.imshow(train_maps_s[0].mean(axis=-1), cmap='gray')
    chosen = set(example_selected[q])
    eligible = set(np.where(example_intervals[:, q, 1] <= desired_thresholds[q])[0].tolist())
    for ridx, (y0, y1, x0, x1) in enumerate(example_coords):
        color = 'lime' if ridx in chosen else ('gold' if ridx in eligible else 'red')
        rect = plt.Rectangle((x0-.5, y0-.5), x1-x0, y1-y0,
                             fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        ax.text(x0, y0, str(ridx), color='white', fontsize=7)
    ax.set_title(
        f'{CLASS_NAMES[q]}\\n'
        f'{len(chosen)}/{len(example_coords)} selected'
    )
    ax.axis('off')

from matplotlib.lines import Line2D
legend_items = [
    Line2D([0], [0], color='lime', lw=3, label='Terpilih'),
    Line2D([0], [0], color='gold', lw=3, label='Lolos gate, tidak final'),
    Line2D([0], [0], color='red', lw=3, label='Gagal desired gate')
]
fig.legend(handles=legend_items, loc='lower center', ncol=3)
plt.suptitle('Modified GASky: desired gate → skyline → batas jumlah region')
plt.tight_layout(rect=[0, .10, 1, .92])
plt.show()

display(pd.DataFrame(example_diag))


## 10. Tuning quantile/cap dan membentuk fitur

Kuantil proximity (0.25/0.50/0.75) dan cap region (0.25/0.50/0.75) dipilih dari validation Macro-F1; tie-break memakai balanced accuracy lalu jumlah region lebih sedikit. TEST tidak masuk ke pencarian ini dan ditransformasikan hanya setelah konfigurasi dipilih. Hyperparameter C/gamma SVM dituning di dalam setiap konfigurasi memakai train/validation yang sama.

In [ ]:
SVM_C_GRID = [0.1, 1.0, 10.0]
# StandardScaler membuat gamma='scale' sebagai pilihan konsisten; hindari grid duplikat.
SVM_GAMMA_GRID = ['scale']
PROXIMITY_QUANTILE_GRID = [0.25, 0.50, 0.75]
MAX_REGION_FRACTION_GRID = [0.25, 0.50, 0.75]


def select_svm_hyperparameters(X_train, y_train, X_val, y_val, model_name):
    rows = []
    for c_value in SVM_C_GRID:
        for gamma_value in SVM_GAMMA_GRID:
            candidate = SVC(kernel='rbf', C=c_value, gamma=gamma_value,
                            class_weight='balanced', decision_function_shape='ovr')
            candidate.fit(X_train, y_train)
            pred = candidate.predict(X_val)
            rows.append({
                'model': model_name, 'C': c_value, 'gamma': gamma_value,
                'validation_macro_f1': f1_score(y_val, pred, average='macro'),
                'validation_balanced_accuracy': balanced_accuracy_score(y_val, pred)
            })
    table = pd.DataFrame(rows).sort_values(
        ['validation_macro_f1', 'validation_balanced_accuracy', 'C'],
        ascending=[False, False, True]
    ).reset_index(drop=True)
    best = table.iloc[0]
    return {'C': float(best['C']), 'gamma': str(best['gamma'])}, table


def transform_dataset(feature_maps, anchors, thresholds, name='dataset',
                      use_gate=True, use_skyline=True, use_cap=True,
                      max_selected_fraction=None):
    features, selected_counts = [], []
    for i, fmap in enumerate(feature_maps):
        feat, counts_i = transform_one_image(
            fmap, anchors, thresholds,
            use_gate=use_gate, use_skyline=use_skyline, use_cap=use_cap,
            max_selected_fraction=max_selected_fraction
        )
        features.append(feat)
        selected_counts.append(counts_i)
        if (i + 1) % 100 == 0 or (i + 1) == len(feature_maps):
            print(f'{name}: {i+1}/{len(feature_maps)}')
    return np.stack(features).astype(np.float32), np.asarray(selected_counts)


# Pilih quantile proximity dan cap hanya dari TRAIN/VALIDATION.
# TEST tidak digunakan selama pencarian konfigurasi ini.
tuning_rows = []
best_validation_key = None
best_gasky_tuning_table = None
best_train_val_transform_seconds = None

for quantile in PROXIMITY_QUANTILE_GRID:
    thresholds_q = {
        q: float(np.quantile(train_desired_distance_samples[q], quantile))
        for q in range(len(CLASS_NAMES))
    }
    for cap_fraction in MAX_REGION_FRACTION_GRID:
        t0 = time.perf_counter()
        Xtr_candidate, counts_tr_candidate = transform_dataset(
            train_maps_s, anchors, thresholds_q,
            f'TRAIN q={quantile:.2f}, cap={cap_fraction:.2f}',
            max_selected_fraction=cap_fraction
        )
        Xval_candidate, counts_val_candidate = transform_dataset(
            val_maps_s, anchors, thresholds_q,
            f'VALIDATION q={quantile:.2f}, cap={cap_fraction:.2f}',
            max_selected_fraction=cap_fraction
        )
        transform_seconds = time.perf_counter() - t0

        candidate_scaler = StandardScaler()
        Xtr_scaled = candidate_scaler.fit_transform(Xtr_candidate)
        Xval_scaled = candidate_scaler.transform(Xval_candidate)
        params, svm_table = select_svm_hyperparameters(
            Xtr_scaled, y_train_aug, Xval_scaled, y_val,
            f'GASky q={quantile:.2f}, cap={cap_fraction:.2f}'
        )
        top = svm_table.iloc[0]
        val_fraction = float(counts_val_candidate.mean() / len(example_coords))
        row = {
            'proximity_quantile': quantile,
            'max_region_fraction': cap_fraction,
            'validation_macro_f1': float(top['validation_macro_f1']),
            'validation_balanced_accuracy': float(top['validation_balanced_accuracy']),
            'mean_validation_region_fraction_selected': val_fraction,
            'selected_C': params['C'], 'selected_gamma': params['gamma'],
            'transform_train_validation_seconds': transform_seconds
        }
        tuning_rows.append(row)
        validation_key = (
            row['validation_macro_f1'],
            row['validation_balanced_accuracy'],
            -row['mean_validation_region_fraction_selected']
        )
        if best_validation_key is None or validation_key > best_validation_key:
            best_validation_key = validation_key
            best_train_val_transform_seconds = transform_seconds
            best_train_gasky = Xtr_candidate
            best_val_gasky = Xval_candidate
            best_train_selected_counts = counts_tr_candidate
            best_val_selected_counts = counts_val_candidate
            desired_thresholds = thresholds_q
            DESIRED_PROXIMITY_QUANTILE = quantile
            MAX_SELECTED_REGION_FRACTION = cap_fraction
            best_gasky_params = params
            best_gasky_tuning_table = svm_table.copy()

        del Xtr_candidate, Xval_candidate, Xtr_scaled, Xval_scaled
        del counts_tr_candidate, counts_val_candidate, candidate_scaler

proximity_cap_tuning_df = pd.DataFrame(tuning_rows).sort_values(
    ['validation_macro_f1', 'validation_balanced_accuracy',
     'mean_validation_region_fraction_selected'],
    ascending=[False, False, True]
).reset_index(drop=True)

# Gunakan konfigurasi pemenang validation untuk semua evaluasi akhir.
X_train_gasky = best_train_gasky
X_val_gasky = best_val_gasky
train_selected_counts = best_train_selected_counts
val_selected_counts = best_val_selected_counts

t0 = time.perf_counter()
X_test_gasky, test_selected_counts = transform_dataset(
    test_maps_s, anchors, desired_thresholds, 'TEST — selected configuration',
    max_selected_fraction=MAX_SELECTED_REGION_FRACTION
)
test_transform_seconds = time.perf_counter() - t0
gasky_transform_seconds = best_train_val_transform_seconds + test_transform_seconds

print('Konfigurasi dipilih dengan validation:', {
    'proximity_quantile': DESIRED_PROXIMITY_QUANTILE,
    'max_region_fraction': MAX_SELECTED_REGION_FRACTION,
    'svm': best_gasky_params
})
display(proximity_cap_tuning_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
pivot = proximity_cap_tuning_df.pivot(
    index='proximity_quantile', columns='max_region_fraction',
    values='validation_macro_f1'
)
im = ax.imshow(pivot.values, cmap='viridis', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns)), labels=[f'{x:.2f}' for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)), labels=[f'{x:.2f}' for x in pivot.index])
ax.set_xlabel('Maksimum fraksi region'); ax.set_ylabel('Quantile proximity')
ax.set_title('Macro-F1 validation untuk tuning quantile × cap')
for y_idx in range(pivot.shape[0]):
    for x_idx in range(pivot.shape[1]):
        ax.text(x_idx, y_idx, f'{pivot.values[y_idx,x_idx]:.3f}',
                ha='center', va='center', color='white')
fig.colorbar(im, ax=ax, label='Validation Macro-F1')
plt.tight_layout(); plt.show()

print('Shapes TRAIN/VALIDATION/TEST:', X_train_gasky.shape, X_val_gasky.shape, X_test_gasky.shape)
print('Waktu transformasi konfigurasi terpilih (train+validation+test):',
      round(gasky_transform_seconds, 2), 'detik')

selection_summary = pd.DataFrame({
    'query': CLASS_NAMES,
    'mean_region_train': train_selected_counts.mean(axis=0),
    'mean_region_validation': val_selected_counts.mean(axis=0),
    'mean_region_test': test_selected_counts.mean(axis=0),
    'total_region': len(example_coords)
})
display(selection_summary.round(3))


### Diagnostik selektivitas dan fallback

Jumlah region final saja belum cukup. Periksa juga proporsi region yang lolos proximity gate, berapa yang tereliminasi oleh skyline/cap, serta frekuensi fallback. Statistik ini membantu menilai apakah threshold terlalu longgar atau terlalu ketat.

In [ ]:
# Diagnostik tahap seleksi pada sampel TRAIN/validation/test.
def collect_selection_diagnostics(feature_maps, anchors, thresholds, name, max_samples=100):
    rows = []
    for sample_idx, fmap in enumerate(feature_maps[:max_samples]):
        _, _, _, _, diag = transform_one_image(fmap, anchors, thresholds, return_details=True)
        for item in diag:
            rows.append({'subset': name, 'sample': sample_idx, **item})
    return pd.DataFrame(rows)

diagnostic_df = pd.concat([
    collect_selection_diagnostics(train_maps_s, anchors, desired_thresholds, 'TRAIN'),
    collect_selection_diagnostics(val_maps_s, anchors, desired_thresholds, 'VALIDATION'),
    collect_selection_diagnostics(test_maps_s, anchors, desired_thresholds, 'TEST')
], ignore_index=True)

diagnostic_summary = diagnostic_df.groupby(['subset', 'query']).agg(
    mean_eligible=('n_eligible', 'mean'),
    mean_skyline_before_cap=('n_skyline_before_cap', 'mean'),
    mean_selected=('n_selected', 'mean'),
    fallback_rate=('fallback_used', 'mean'),
    cap_rate=('cap_applied', 'mean')
).reset_index()
display(diagnostic_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for subset, part in diagnostic_summary.groupby('subset'):
    axes[0].plot(part['query'], part['mean_selected'], marker='o', label=subset)
    axes[1].plot(part['query'], part['fallback_rate'], marker='o', label=subset)
axes[0].set_title('Rata-rata region final'); axes[0].set_ylabel('Jumlah region')
axes[1].set_title('Fallback rate'); axes[1].set_ylabel('Proporsi')
for ax in axes:
    ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()


## 11. StandardScaler dan SVM-RBF

In [ ]:
# Scaler dan model final fit pada TRAIN menggunakan konfigurasi terpilih dari validation.
feature_scaler = StandardScaler()
X_train_svm = feature_scaler.fit_transform(X_train_gasky)
X_val_svm = feature_scaler.transform(X_val_gasky)
X_test_svm = feature_scaler.transform(X_test_gasky)

gasky_params = best_gasky_params
gasky_tuning_table = best_gasky_tuning_table.copy()
svm = SVC(kernel='rbf', class_weight='balanced', decision_function_shape='ovr',
          random_state=SEED, **gasky_params)
t_fit = time.perf_counter()
svm.fit(X_train_svm, y_train_aug)
gasky_svm_fit_seconds = time.perf_counter() - t_fit
print('SVM Modified GASky final fit; parameter dipilih dari validation:', gasky_params)


## 12. Baseline pembanding: EfficientNetB0 → GAP → SVM-RBF

In [ ]:
# Baseline memakai feature map backbone yang sama, tanpa GASky.
Xb_train = train_maps.mean(axis=(1,2))
Xb_val = val_maps.mean(axis=(1,2))
Xb_test = test_maps.mean(axis=(1,2))

baseline_scaler = StandardScaler()
Xb_train_s = baseline_scaler.fit_transform(Xb_train)
Xb_val_s = baseline_scaler.transform(Xb_val)
Xb_test_s = baseline_scaler.transform(Xb_test)

baseline_params, baseline_tuning_table = select_svm_hyperparameters(
    Xb_train_s, y_train_aug, Xb_val_s, y_val, 'Baseline GAP'
)
baseline_svm = SVC(kernel='rbf', class_weight='balanced', decision_function_shape='ovr',
                   random_state=SEED, **baseline_params)
t_fit = time.perf_counter()
baseline_svm.fit(Xb_train_s, y_train_aug)
baseline_svm_fit_seconds = time.perf_counter() - t_fit
print('Parameter baseline terpilih dari validation:', baseline_params)
display(baseline_tuning_table.round(4))


## 13. Evaluasi validation dan test

In [ ]:
def predict_and_report(name, model, X, y_true):
    pred = model.predict(X)
    print(f'\n===== {name} =====')
    print('Accuracy          :', round(accuracy_score(y_true,pred),4))
    print('Balanced accuracy :', round(balanced_accuracy_score(y_true,pred),4))
    print('Macro-F1          :', round(f1_score(y_true,pred,average='macro'),4))
    print(classification_report(y_true, pred, labels=range(len(CLASS_NAMES)),
                                target_names=CLASS_NAMES, zero_division=0, digits=4))
    return pred

# Validation sudah digunakan untuk memilih C/gamma; metrik validation bersifat tuning.
pred_val_g = predict_and_report('Modified GASky — VALIDATION (tuning)', svm, X_val_svm, y_val)
pred_val_b = predict_and_report('Baseline GAP — VALIDATION (tuning)', baseline_svm, Xb_val_s, y_val)
# Test hanya dipakai setelah parameter dipilih.
pred_test_g = predict_and_report('Modified GASky — TEST (final)', svm, X_test_svm, y_test)
pred_test_b = predict_and_report('Baseline GAP — TEST (final)', baseline_svm, Xb_test_s, y_test)

results = []
for split_name, yt, pg, pb in [
    ('Validation (used for tuning)', y_val, pred_val_g, pred_val_b),
    ('Test (final)', y_test, pred_test_g, pred_test_b)]:
    for model_name, pred in [('Modified GASky + SVM', pg), ('Baseline GAP + SVM', pb)]:
        results.append({
            'subset': split_name, 'model': model_name,
            'accuracy': accuracy_score(yt,pred),
            'balanced_accuracy': balanced_accuracy_score(yt,pred),
            'macro_f1': f1_score(yt,pred,average='macro')
        })
results_df = pd.DataFrame(results)
display(results_df.round(4))

# Interval ketidakpastian bootstrap berstrata kelas untuk TEST (unit resampling = citra).
# Ini bukan CI berkelompok pasien; metadata patient ID tidak tersedia.
def stratified_test_bootstrap(y_true, y_pred, n_bootstrap=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.unique(y_true)
    class_indices = {k: np.where(y_true == k)[0] for k in labels}
    rows = []
    for _ in range(n_bootstrap):
        sampled = np.concatenate([
            rng.choice(class_indices[k], size=len(class_indices[k]), replace=True)
            for k in labels
        ])
        yt_b, yp_b = y_true[sampled], y_pred[sampled]
        rows.append({
            'macro_f1': f1_score(yt_b, yp_b, labels=range(len(CLASS_NAMES)),
                                 average='macro', zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(yt_b, yp_b),
            'accuracy': accuracy_score(yt_b, yp_b)
        })
    boot = pd.DataFrame(rows)
    return pd.DataFrame({
        metric: [float(boot[metric].quantile(.025)),
                 float(boot[metric].quantile(.975))]
        for metric in boot.columns
    }, index=['ci_2.5%', 'ci_97.5%']).T

bootstrap_ci = pd.concat({
    'Modified GASky + SVM': stratified_test_bootstrap(y_test, pred_test_g),
    'Baseline GAP + SVM': stratified_test_bootstrap(y_test, pred_test_b)
}, names=['model','metric']).reset_index()
print('95% bootstrap CI pada TEST (image-level, stratified by class):')
display(bootstrap_ci.round(4))


## 14. Visualisasi confusion matrix dan metrik per kelas

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=30)
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Label aktual'); ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
    fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

plot_confusion(y_test, pred_test_g, 'Confusion Matrix — Modified GASky + SVM (Test)')
plot_confusion(y_test, pred_test_b, 'Confusion Matrix — Baseline GAP + SVM (Test)')

p, r, f, s = precision_recall_fscore_support(y_test, pred_test_g,
                                               labels=range(len(CLASS_NAMES)), zero_division=0)
per_class = pd.DataFrame({'precision':p,'recall':r,'f1_score':f,'support':s}, index=CLASS_NAMES)
display(per_class.round(4))
ax = per_class[['precision','recall','f1_score']].plot(kind='bar', figsize=(9,4))
ax.set_title('Metrik per kelas — Modified GASky (Test)')
ax.set_xlabel('Kelas'); ax.set_ylabel('Skor'); ax.set_ylim(0,1)
ax.grid(axis='y', alpha=.25); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 15. Ablation study: kontribusi komponen selector

Semua ablation memakai backbone, split, augmentasi TRAIN, dan aturan pemilihan C/gamma yang sama. Setiap metode dituning di validation lalu dilaporkan pada test. Metode yang diuji sudah ditetapkan sebelumnya; jangan memilih metode final berdasarkan skor test. Perbandingan ini menguji kontribusi gate, skyline, cap, dan pooling region.

Indikator dukungan terhadap usulan: full method perlu mengungguli GAP pada Macro-F1 test dan memperbaiki dibandingkan ablation yang menghapus tahap yang diklaim penting, tanpa menurunkan recall kelas secara bermakna. Baca juga 95% CI, dimensi fitur, waktu, dan support vector. Jika tidak, simpulkan bahwa konfigurasi ini belum menunjukkan keunggulan; jangan mengubah keputusan berdasarkan skor test.


In [ ]:
ablation_rows = []

def add_ablation_result(name, model, y_pred, feature_dimension,
                        transform_seconds=np.nan, selected_fraction=np.nan):
    ablation_rows.append({
        'method': name,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'test_macro_f1': f1_score(y_test, y_pred, average='macro'),
        'feature_dimension': int(feature_dimension),
        'dimension_ratio_vs_GAP': float(feature_dimension / Xb_train.shape[1]),
        'mean_region_fraction_selected': selected_fraction,
        'transform_seconds_train_val_test': transform_seconds,
        'n_support_vectors': int(model.n_support_.sum())
    })


def fit_score_ablation(name, Xtr, Xv, Xte, transform_seconds,
                       selected_fraction=np.nan):
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xv_s = scaler.transform(Xv)
    Xte_s = scaler.transform(Xte)
    params, _ = select_svm_hyperparameters(Xtr_s, y_train_aug, Xv_s, y_val, name)
    model = SVC(kernel='rbf', class_weight='balanced', decision_function_shape='ovr',
                random_state=SEED, **params)
    t0 = time.perf_counter()
    model.fit(Xtr_s, y_train_aug)
    fit_seconds = time.perf_counter() - t0
    pred = model.predict(Xte_s)
    add_ablation_result(name, model, pred, Xtr.shape[1], transform_seconds, selected_fraction)
    ablation_rows[-1]['svm_fit_seconds'] = fit_seconds
    ablation_rows[-1]['selected_C'] = params['C']
    ablation_rows[-1]['selected_gamma'] = params['gamma']


# Existing end-to-end baselines.
add_ablation_result('EfficientNet GAP', baseline_svm, pred_test_b, Xb_train.shape[1])
ablation_rows[-1]['svm_fit_seconds'] = baseline_svm_fit_seconds
ablation_rows[-1]['selected_C'] = baseline_params['C']
ablation_rows[-1]['selected_gamma'] = baseline_params['gamma']
full_fraction = float(np.concatenate([
    train_selected_counts, val_selected_counts, test_selected_counts
]).mean() / len(example_coords))
add_ablation_result('Full: gate + skyline + cap', svm, pred_test_g,
                    X_train_gasky.shape[1], gasky_transform_seconds, full_fraction)
ablation_rows[-1]['svm_fit_seconds'] = gasky_svm_fit_seconds
ablation_rows[-1]['selected_C'] = gasky_params['C']
ablation_rows[-1]['selected_gamma'] = gasky_params['gamma']

# Region pooling control and matched-dimension ANOVA feature selection.
def pool_all_regions(feature_maps):
    return np.concatenate([
        feature_maps.mean(axis=(1,2)), feature_maps.max(axis=(1,2))
    ], axis=1).astype(np.float32)

t0 = time.perf_counter()
P_train = pool_all_regions(train_maps)
P_val = pool_all_regions(val_maps)
P_test = pool_all_regions(test_maps)
pool_seconds = time.perf_counter() - t0
fit_score_ablation('All-region mean+max (no selection)', P_train, P_val, P_test, pool_seconds)

filter_t0 = time.perf_counter()
k_filter = min(int(Xb_train.shape[1]), int(P_train.shape[1]))
filter_selector = SelectKBest(score_func=f_classif, k=k_filter)
P_train_k = filter_selector.fit_transform(P_train, y_train_aug)
P_val_k = filter_selector.transform(P_val)
P_test_k = filter_selector.transform(P_test)
filter_transform_seconds = time.perf_counter() - filter_t0
fit_score_ablation(f'All-region + ANOVA top-{k_filter}', P_train_k, P_val_k, P_test_k,
                   pool_seconds + filter_transform_seconds)
del P_train, P_val, P_test, P_train_k, P_val_k, P_test_k

# Isolate skyline alone, gate alone, and gate+skyline without cap.
selector_specs = [
    ('Interval skyline only', dict(use_gate=False, use_skyline=True, use_cap=False)),
    ('Proximity gate only', dict(use_gate=True, use_skyline=False, use_cap=False)),
    ('Gate + skyline (no cap)', dict(use_gate=True, use_skyline=True, use_cap=False)),
]
for method_name, options in selector_specs:
    t0 = time.perf_counter()
    A_train, c_train = transform_dataset(train_maps_s, anchors, desired_thresholds,
                                         method_name+' TRAIN', **options)
    A_val, c_val = transform_dataset(val_maps_s, anchors, desired_thresholds,
                                     method_name+' VALIDATION', **options)
    A_test, c_test = transform_dataset(test_maps_s, anchors, desired_thresholds,
                                       method_name+' TEST', **options)
    elapsed = time.perf_counter() - t0
    fraction = float(np.concatenate([c_train,c_val,c_test]).mean()/len(example_coords))
    fit_score_ablation(method_name, A_train, A_val, A_test, elapsed, fraction)
    del A_train, A_val, A_test

ablation_df = pd.DataFrame(ablation_rows).sort_values(
    'test_macro_f1', ascending=False
).reset_index(drop=True)
display(ablation_df.round(4))
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(ablation_df['method'][::-1], ablation_df['test_macro_f1'][::-1])
ax.set_xlabel('Test Macro-F1'); ax.set_title('Ablation: test Macro-F1 (baca bersama biaya dan dimensi)')
ax.grid(axis='x', alpha=.25); plt.tight_layout(); plt.show()


## 16. Pengukuran latensi inferensi per citra

Latensi dipisahkan dari waktu training/ekstraksi batch. Benchmark ini mengukur jalur satu citra: EfficientNetB0 → transformasi fitur (Modified GASky atau GAP) → scaling → prediksi SVM. Beberapa citra test dipakai untuk timing tanpa mengubah model; laporkan perangkat (CPU/GPU), median, dan p95. Nilai ini bergantung pada runtime Colab dan bukan estimasi waktu training.

In [ ]:
import time

def benchmark_single_image_latency(n_samples=30, repeats=3):
    n = min(n_samples, len(X_test_img))
    indices = np.linspace(0, len(X_test_img)-1, n, dtype=int)
    # Warm-up menghindari pencatatan inisialisasi graph/kernel pertama sebagai inferensi normal.
    warm = X_test_img[indices[0]:indices[0]+1].astype(np.float32)
    _ = feature_extractor(warm, training=False).numpy()
    _ = svm.predict(X_test_svm[:1])
    rows = []
    for idx in indices:
        image = X_test_img[idx:idx+1].astype(np.float32)
        for rep in range(repeats):
            t0 = time.perf_counter()
            fmap = feature_extractor(image, training=False).numpy()[0].astype(np.float32)
            t_backbone = time.perf_counter()
            fmap_s = apply_channel_scaler(fmap[None, ...], channel_scaler)[0]
            feat, _ = transform_one_image(fmap_s, anchors, desired_thresholds)
            t_gasky = time.perf_counter()
            x_s = feature_scaler.transform(feat.reshape(1, -1))
            _ = svm.predict(x_s)
            t_g_end = time.perf_counter()
            # Baseline shares the exact same image and backbone extraction.
            x_base = baseline_scaler.transform(fmap.mean(axis=(0,1)).reshape(1,-1))
            _ = baseline_svm.predict(x_base)
            t_b_end = time.perf_counter()
            rows.append({
                'sample': int(idx), 'repeat': rep+1,
                'backbone_ms': (t_backbone-t0)*1000,
                'gasky_transform_ms': (t_gasky-t_backbone)*1000,
                'gasky_total_ms': (t_g_end-t0)*1000,
                'baseline_post_backbone_ms': (t_b_end-t_backbone)*1000,
                'baseline_total_ms': (t_b_end-t0)*1000
            })
    raw = pd.DataFrame(rows)
    cols = ['backbone_ms','gasky_transform_ms','gasky_total_ms',
            'baseline_post_backbone_ms','baseline_total_ms']
    summary = pd.DataFrame({
        'median_ms': raw[cols].median(),
        'p95_ms': raw[cols].quantile(0.95)
    })
    summary['median_images_per_second'] = 1000 / summary['median_ms']
    return raw, summary.reset_index(names='stage')

latency_raw, latency_summary = benchmark_single_image_latency()
device = 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'
print(f'Perangkat benchmark: {device}; {min(30, len(X_test_img))} citra test × 3 pengulangan')
display(latency_summary.round(3))
plt.figure(figsize=(9,4))
plt.bar(latency_summary['stage'], latency_summary['median_ms'],
        yerr=latency_summary['p95_ms']-latency_summary['median_ms'], capsize=4)
plt.ylabel('Latensi (ms/citra)'); plt.title('Latensi inferensi satu citra (median, whisker p95)')
plt.xticks(rotation=20, ha='right'); plt.grid(axis='y', alpha=.25); plt.tight_layout(); plt.show()


## 17. Simpan artefak eksperimen


In [ ]:
# Simpan model, scaler, anchor, konfigurasi, dan ringkasan metrik.
artifacts = {
    'backbone': 'EfficientNetB0_ImageNet_frozen',
    'classes': CLASS_NAMES,
    'image_size': IMAGE_SIZE,
    'region_size': (REGION_H, REGION_W),
    'modified_gasky': {
        'desired_proximity_quantile': DESIRED_PROXIMITY_QUANTILE,
        'desired_thresholds_train_only': desired_thresholds,
        'max_selected_region_fraction': MAX_SELECTED_REGION_FRACTION,
        'dominance': 'interval dominance applied only after desired proximity gate',
        'fallback': 'select nearest region by desired d_max if no region passes gate'
    },
    'channel_scaler': channel_scaler,
    'anchors': anchors,
    'feature_scaler': feature_scaler,
    'svm': svm,
    'baseline_scaler': baseline_scaler,
    'baseline_svm': baseline_svm,
    'selected_svm_params': {'modified_gasky': gasky_params, 'baseline_gap': baseline_params},
    'split_protocol': 'image-level stratified 70:15:15; patient IDs unavailable',
    'augmentation': {'train_only': True, 'max_multiplier': AUGMENT_MAX_MULTIPLIER}
}
joblib.dump(artifacts, OUTPUT_DIR / 'modified_gasky_svm_artifacts.joblib')
results_df.to_csv(OUTPUT_DIR / 'metrics_comparison.csv', index=False)
per_class.to_csv(OUTPUT_DIR / 'per_class_test_metrics.csv')
selection_summary.to_csv(OUTPUT_DIR / 'selected_region_summary.csv', index=False)
diagnostic_summary.to_csv(OUTPUT_DIR / 'selection_diagnostics.csv', index=False)
latency_summary.to_csv(OUTPUT_DIR / 'inference_latency_summary.csv', index=False)
latency_raw.to_csv(OUTPUT_DIR / 'inference_latency_raw.csv', index=False)
gasky_tuning_table.to_csv(OUTPUT_DIR / 'gasky_svm_validation_tuning.csv', index=False)
baseline_tuning_table.to_csv(OUTPUT_DIR / 'baseline_svm_validation_tuning.csv', index=False)
proximity_cap_tuning_df.to_csv(OUTPUT_DIR / 'proximity_cap_validation_sweep.csv', index=False)
ablation_df.to_csv(OUTPUT_DIR / 'ablation_comparison.csv', index=False)
bootstrap_ci.to_csv(OUTPUT_DIR / 'test_bootstrap_intervals.csv', index=False)

summary = {
    'n_original_images_after_exact_dedup': int(len(data)),
    'n_train_original': int(len(train_df)),
    'n_validation_original': int(len(val_df)),
    'n_test_original': int(len(test_df)),
    'n_train_after_augmentation': int(len(X_train_aug)),
    'feature_map_shape': list(train_maps.shape[1:]),
    'gasky_feature_dimension': int(X_train_gasky.shape[1]),
    'desired_proximity_quantile': float(DESIRED_PROXIMITY_QUANTILE),
    'desired_thresholds_train_only': {CLASS_NAMES[k]: float(desired_thresholds[k]) for k in range(len(CLASS_NAMES))},
    'max_selected_region_fraction': float(MAX_SELECTED_REGION_FRACTION),
    'feature_dimensions': {'modified_gasky': int(X_train_gasky.shape[1]), 'baseline_gap': int(Xb_train.shape[1])},
    'selected_svm_params': {'modified_gasky': gasky_params, 'baseline_gap': baseline_params},
    'latency_device': device,
    'latency_summary_ms': latency_summary.to_dict(orient='records'),
    'split': 'image-level stratified 70:15:15',
    'patient_id_available': False,
    'limitation': 'Patient-independent evaluation cannot be guaranteed; possible patient-level leakage.'
}
with open(OUTPUT_DIR / 'experiment_summary.json','w') as f:
    json.dump(summary,f,indent=2)
print('Artefak tersimpan di:', OUTPUT_DIR)
print(json.dumps(summary, indent=2))


## 18. Catatan metodologis dan interpretasi

1. Dataset tidak menyediakan patient ID. Hasil hanya mendukung klaim evaluasi **image-level stratified split 70:15:15**; kebocoran antar-citra pasien yang sama masih mungkin. SHA-256 hanya mendeteksi file identik byte-per-byte.
2. Augmentasi hanya diterapkan setelah split dan pada TRAIN. Validation/test tetap citra asli.
3. Channel scaler, anchor kelas, dan ambang proximity dikalibrasi dari TRAIN. Validation dipakai memilih C/gamma SVM; karena itu skor validation adalah skor tuning. TEST dipakai untuk pelaporan akhir setelah pilihan ditetapkan.
4. Proximity gate `d_max ≤ τ_q`, interval dominance, fallback, dan batas jumlah region adalah **modifikasi usulan**. Jangan menyebutnya sebagai bagian identik dari GASky asli. Evaluasi harus melaporkan fallback/cap rate serta jumlah region.
5. Laporkan Macro-F1, balanced accuracy, confusion matrix, metrik per kelas, dimensi fitur, selektivitas region, dan latensi median/p95 bersama perangkat/runtime. Perbandingan GAP vs GASky perlu memakai split, backbone, dan kondisi timing yang sama.
6. Hasil dari satu image-level split masih memiliki ketidakpastian tinggi. Jika metadata pasien tersedia, evaluasi lanjutan sebaiknya menggunakan patient-level split atau cross-validation berkelompok; bila tidak tersedia, nyatakan batasan ini dengan jelas.
7. Klaim kontribusi harus dibatasi pada modifikasi dan evaluasi yang benar-benar dilakukan. Performa harus ditentukan dari hasil run Colab, bukan dari rancangan kode saja.

8. Quantile/cap dan C/gamma dipilih memakai validation. Test dipakai untuk satu evaluasi akhir dan ablation yang sudah ditentukan sebelumnya; jangan memilih model final dari tabel test.
9. Bootstrap 95% CI pada notebook di-resample per citra dan berstrata kelas. Karena patient ID tidak tersedia, interval ini tidak menangkap korelasi antar citra pasien dan bukan pengganti evaluasi patient-level.
10. Grid tuning terdiri dari 9 pasangan quantile×cap; hasil validasi dapat sedikit optimistis karena beberapa konfigurasi dicoba. Replikasi berkelompok atau test eksternal tetap dibutuhkan untuk klaim generalisasi yang lebih kuat.
